# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

**Збережений прогін.** Повне донавчання і перевірки виконано локально на RTX 3050 на 3668 тренувальних і 408 валідаційних парах MRPC. У Colab цей самий ноутбук можна запустити на T4.

In [1]:
from time import perf_counter
from IPython import get_ipython

lab5_shell = get_ipython()
lab5_timing = []
previous = globals().get("lab5_cell_hooks")
if previous is not None:
    for event, callback in previous:
        lab5_shell.events.unregister(event, callback)

def lab5_before_cell(info):
    global lab5_start
    lab5_start = perf_counter()

def lab5_after_cell(result):
    if "lab5_start" not in globals():
        return
    seconds = perf_counter() - lab5_start
    source = next((line.strip() for line in result.info.raw_cell.splitlines() if line.strip()), "")
    error = result.error_before_exec or result.error_in_exec
    lab5_timing.append((result.execution_count, source[:76], seconds,
                        type(error).__name__ if error else "OK"))

lab5_cell_hooks = (
    ("pre_run_cell", lab5_before_cell), ("post_run_cell", lab5_after_cell),
)
for event, callback in lab5_cell_hooks:
    lab5_shell.events.register(event, callback)

In [ ]:
%pip install -q "transformers==4.57.3" "datasets>=4,<5" "accelerate>=1,<2" scikit-learn

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

/tmp/ai_systems_lab5_gpu/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

if not torch.cuda.is_available():
    torch.set_num_threads(min(6, torch.get_num_threads()))

Пристрій: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [4]:
import os

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
if os.getenv("LAB5_CPU_PILOT") == "1":
    for split, limit in (("train", 128), ("validation", 96)):
        raw_datasets[split] = raw_datasets[split].shuffle(seed=42).select(range(limit))
    print("Скорочений локальний CPU-прогін: 128 train, 96 validation.")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [6]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Device set to use cuda:0


[{'score': 0.09747544676065445,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.05238320305943489,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627117812633514,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.03788600489497185,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.0376809760928154,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [7]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981506049633026,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15974067151546478,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547306180000305,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.037968847900629044,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423840507864952,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [8]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [9]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:  27%|██▋       | 1000/3668 [00:00<00:00, 7465.27 examples/s]

Map:  55%|█████▍    | 2000/3668 [00:00<00:00, 7550.94 examples/s]

Map:  82%|████████▏ | 3000/3668 [00:00<00:00, 7468.08 examples/s]

Map: 100%|██████████| 3668/3668 [00:00<00:00, 7413.69 examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map: 100%|██████████| 408/408 [00:00<00:00, 6841.44 examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [10]:
set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Linear(in_features=768, out_features=2, bias=True)


In [11]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=10,
    disable_tqdm=True,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [13]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
three_epoch_result = trainer.train()
three_epoch_metrics = trainer.evaluate()
three_epoch_metrics

Параметрів, що тренуються: 109,483,778


{'loss': 0.65, 'grad_norm': 1.5428423881530762, 'learning_rate': 1.973913043478261e-05, 'epoch': 0.043478260869565216}


{'loss': 0.6557, 'grad_norm': 3.83892560005188, 'learning_rate': 1.9449275362318843e-05, 'epoch': 0.08695652173913043}


{'loss': 0.6218, 'grad_norm': 4.246549606323242, 'learning_rate': 1.9159420289855074e-05, 'epoch': 0.13043478260869565}


{'loss': 0.6204, 'grad_norm': 7.645400047302246, 'learning_rate': 1.8869565217391305e-05, 'epoch': 0.17391304347826086}


{'loss': 0.6189, 'grad_norm': 3.028064250946045, 'learning_rate': 1.857971014492754e-05, 'epoch': 0.21739130434782608}


... (65 outputs left out)

{'loss': 0.1615, 'grad_norm': 2.310441493988037, 'learning_rate': 2.8985507246376815e-08, 'epoch': 3.0}


{'eval_loss': 0.4508305788040161, 'eval_accuracy': 0.8455882352941176, 'eval_f1': 0.8888888888888888, 'eval_runtime': 0.6727, 'eval_samples_per_second': 606.521, 'eval_steps_per_second': 10.406, 'epoch': 3.0}
{'train_runtime': 99.1606, 'train_samples_per_second': 110.971, 'train_steps_per_second': 6.958, 'train_loss': 0.34149593132129613, 'epoch': 3.0}


{'eval_loss': 0.4508305788040161, 'eval_accuracy': 0.8455882352941176, 'eval_f1': 0.8888888888888888, 'eval_runtime': 0.6726, 'eval_samples_per_second': 606.569, 'eval_steps_per_second': 10.407, 'epoch': 3.0}


{'eval_loss': 0.4508305788040161,
 'eval_accuracy': 0.8455882352941176,
 'eval_f1': 0.8888888888888888,
 'eval_runtime': 0.6726,
 'eval_samples_per_second': 606.569,
 'eval_steps_per_second': 10.407,
 'epoch': 3.0}

## 4. Перевірка на власних прикладах

In [14]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
    ("The meeting starts at eight.", "The meeting starts at nine."),
    ("No dogs were allowed inside.", "Dogs were permitted in the building."),
    ("The flight was delayed by two hours.", "The plane departed two hours late."),
    ("The road closed after the storm.", "After the storm, the road was shut."),
]
expected = [1, 0, 1, 0, 0, 1, 1]
for (s1, s2), label in zip(pairs, expected):
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}; еталон: {label}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 81.5%; еталон: 1

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.5%; еталон: 0

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 98.7%; еталон: 1

The meeting starts at eight.
The meeting starts at nine.
→ парафраз: 98.0%; еталон: 0

No dogs were allowed inside.
Dogs were permitted in the building.
→ парафраз: 95.6%; еталон: 0

The flight was delayed by two hours.
The plane departed two hours late.
→ парафраз: 94.0%; еталон: 1

The road closed after the storm.
After the storm, the road was shut.
→ парафраз: 98.8%; еталон: 1



## 0.1. Теоретичні питання 1–10

**1. Час і проблеми запуску.** Повний MRPC виконано локально на RTX 3050. Клітинки разом зайняли **299,75 с**. Безпосереднє тренування разом з оцінюванням по епохах тривало **99,2 с** для 3 епох і **179,3 с** для 5. У збереженому прогоні помилок виконання не було. Під час налаштування середовища завантаження CUDA-бібліотеки спочатку обірвалось через таймаут, після повтору встановлення завершилось. Попередження BERT про нову голову очікуване.

**2. Токен [CLS].** Це службовий токен на початку послідовності. На кожному шарі його прихований вектор оновлюється через self-attention, тому може враховувати інформацію з обох речень. Голова отримує pooled output, побудований із фінального вектора [CLS]. Сам токен є входом, а представлення для класифікації формується шарами й пристосовується до задачі під час донавчання.

**3. Пряме застосування BERT.** `bert-base-uncased` із початковою MLM-головою прогнозує пропущений токен у двонапрямному англомовному контексті, як у `fill-mask`. Під час pretraining також використовувалась задача NSP. Для оцінки парафраз додається окремий класифікатор. [Картка BERT](https://huggingface.co/google-bert/bert-base-uncased)

**4. Сексизм і модель.** Варто вимірювати та зменшувати стереотипні зміщення. У фактичному прикладі перші прогнози для «man» і «woman» — `carpenter` (9,75%) та `nurse` (21,98%). Порівнюю топ-k професій і їхні ймовірності в парах речень, де змінюється лише гендер. Я б розширив тест різними ролями та контекстами, перевірив навчальні дані, збалансував або доповнив їх контрфактичними прикладами й повторно виміряв зміщення. Одна пара демонструє можливу проблему, а висновок про масштаб потребує більшого набору.

**5. Числа 0 і 1.** Це `token_type_ids`, ідентифікатори сегментів. [CLS], перше речення та його [SEP] належать сегменту 0. Друге речення й фінальний [SEP] належать сегменту 1. BERT додає відповідне сегментне вбудовування до токенного й позиційного.

**6. Числа 768 і 2.** 768 — розмір вектора, який отримує лінійна голова BERT Base. Два виходи — логіти для «не парафраз» і «парафраз». У голові `768 × 2 + 2 = 1538` параметрів із bias.

**7. Співвідношення параметрів.** Більшість параметрів у кістяку, де вже є мовні представлення. Для двох класів це корисно, бо новій голові потрібно вивчити лише межу рішення. Повне донавчання все одно оновлює кістяк, тож він визначає витрати пам'яті та ризик перенавчання.

**8. Accuracy і F1.** Accuracy — частка пар з правильною міткою. Бінарна F1 — гармонічне середнє precision і recall для позитивного класу 1. Висока accuracy можлива, якщо модель часто передбачає поширений клас. F1 показує баланс пропущених і хибно знайдених парафраз.

**9. Більше епох.** Порівнюю 3 і 5 епох з однаковими checkpoint, seed, поділом і гіперпараметрами. Після повторного старту оптимізатора горизонти лінійного розкладу LR різні, тому порівнюю окремі запуски. Якщо train loss спадає, а validation loss зростає чи F1 перестає рости, додаткові епохи можуть перенавчати модель. На повному MRPC train loss зменшився з **0,3415** до **0,2309**, але фінальний validation loss зріс з **0,4508** до **0,6935**, а F1 змінилася з **0,889** до **0,892**. У п’яти-епоховому запуску найкраща F1 була **0,894** на третій епосі, а мінімум validation loss — **0,3762** на другій. Після неї модель краще підганяла тренувальні дані й давала гірші валідаційні ймовірності, хоча F1 майже не змінилася. Це аргумент на користь вибору стану за валідацією. Вивід нижче містить оцінки по епохах.

**10. Власні перевірки.** До §4 додано чотири пари з еталонами. Дві з однаковими ключовими словами змінюють час або заперечення. Дві інші перефразовують затримку рейсу й перекриття дороги. За порога 50% BERT правильно розпізнав дві додані парафрази, а обидва непарафрази класифікував хибно. Він оцінив різний час зустрічі в 98,0% і пару зі зміненим запереченням у 95,6% як парафрази. Лексична схожість тут створює проблеми для розрізнення змісту.

In [15]:
from dataclasses import replace

# Перший прогін уже виміряно; звільняємо його GPU-стан перед другим.
if torch.cuda.is_available():
    trainer.model.to("cpu")
    trainer.optimizer = None
    trainer.lr_scheduler = None
    trainer.accelerator.free_memory()

set_seed(42)
model_5 = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
trainer_5 = Trainer(
    model=model_5,
    args=replace(training_args, output_dir="bert-mrpc-5", num_train_epochs=5),
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
five_epoch_result = trainer_5.train()
five_epoch_metrics = trainer_5.evaluate()

from IPython.display import Markdown, display
table = ["| Запуск | Епоха | Validation loss | Accuracy | F1 |", "|---|---:|---:|---:|---:|"]
for name, run in (("3 епохи", trainer), ("5 епох", trainer_5)):
    for entry in run.state.log_history[:-1]:
        if "eval_loss" in entry:
            table.append(
                f"| {name} | {entry['epoch']:.0f} | {entry['eval_loss']:.4f} | "
                f"{entry['eval_accuracy']:.3f} | {entry['eval_f1']:.3f} |"
            )
display(Markdown("\n".join(table)))
for name, result, scores in (
    ("3 епохи", three_epoch_result, three_epoch_metrics),
    ("5 епох", five_epoch_result, five_epoch_metrics),
):
    print(f"{name} — train loss {result.training_loss:.4f}, "
          f"валідаційна F1 {scores['eval_f1']:.3f}, "
          f"час тренування {result.metrics['train_runtime']:.1f} с.")
delta = five_epoch_metrics["eval_f1"] - three_epoch_metrics["eval_f1"]
print(f"Зміна F1 після переходу з 3 на 5 епох: {delta:+.3f}.")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.6499, 'grad_norm': 1.5432614088058472, 'learning_rate': 1.9843478260869567e-05, 'epoch': 0.043478260869565216}


{'loss': 0.6557, 'grad_norm': 3.83821702003479, 'learning_rate': 1.9669565217391307e-05, 'epoch': 0.08695652173913043}


{'loss': 0.6217, 'grad_norm': 4.227650165557861, 'learning_rate': 1.9495652173913044e-05, 'epoch': 0.13043478260869565}


{'loss': 0.6204, 'grad_norm': 7.672570705413818, 'learning_rate': 1.9321739130434785e-05, 'epoch': 0.17391304347826086}


{'loss': 0.6187, 'grad_norm': 2.9996697902679443, 'learning_rate': 1.9147826086956522e-05, 'epoch': 0.21739130434782608}


... (114 outputs left out)

{'eval_loss': 0.6935346722602844, 'eval_accuracy': 0.8480392156862745, 'eval_f1': 0.8923611111111112, 'eval_runtime': 0.7628, 'eval_samples_per_second': 534.847, 'eval_steps_per_second': 9.176, 'epoch': 5.0}
{'train_runtime': 179.3124, 'train_samples_per_second': 102.28, 'train_steps_per_second': 6.413, 'train_loss': 0.23090911720110022, 'epoch': 5.0}


{'eval_loss': 0.6935346722602844, 'eval_accuracy': 0.8480392156862745, 'eval_f1': 0.8923611111111112, 'eval_runtime': 0.7794, 'eval_samples_per_second': 523.45, 'eval_steps_per_second': 8.981, 'epoch': 5.0}


| Запуск | Епоха | Validation loss | Accuracy | F1 |
|---|---:|---:|---:|---:|
| 3 епохи | 1 | 0.4130 | 0.814 | 0.862 |
| 3 епохи | 2 | 0.3692 | 0.848 | 0.892 |
| 3 епохи | 3 | 0.4508 | 0.846 | 0.889 |
| 5 епох | 1 | 0.3994 | 0.819 | 0.869 |
| 5 епох | 2 | 0.3762 | 0.843 | 0.890 |
| 5 епох | 3 | 0.5334 | 0.853 | 0.894 |
| 5 епох | 4 | 0.6876 | 0.841 | 0.889 |
| 5 епох | 5 | 0.6935 | 0.848 | 0.892 |

3 епохи — train loss 0.3415, валідаційна F1 0.889, час тренування 99.2 с.
5 епох — train loss 0.2309, валідаційна F1 0.892, час тренування 179.3 с.
Зміна F1 після переходу з 3 на 5 епох: +0.003.


In [16]:
from IPython.display import Markdown, display

timing = ["| № | Клітинка | Час, с | Статус |", "|---:|---|---:|---|"]
for count, first_line, seconds, status in lab5_timing:
    first_line = first_line.replace("|", " / ").replace(chr(96), "")
    timing.append(f"| {count} | {first_line} | {seconds:.2f} | {status} |")
display(Markdown("\n".join(timing)))
print(f"Сумарний час усіх зафіксованих клітинок: {sum(row[2] for row in lab5_timing):.2f} с.")

| № | Клітинка | Час, с | Статус |
|---:|---|---:|---|
| 2 | import numpy as np | 12.06 | OK |
| 3 | # Фіксуємо seed, щоб результати повторювалися між запусками | 0.15 | OK |
| 4 | import os | 2.37 | OK |
| 5 | # Подивимося на один приклад | 0.00 | OK |
| 6 | from transformers import pipeline | 1.91 | OK |
| 7 | print("Не по темі, але погляньте, яка \"несправедливість\"...") | 0.01 | OK |
| 8 | checkpoint = "bert-base-uncased" | 0.78 | OK |
| 9 | def tokenize_function(batch): | 0.58 | OK |
| 10 | set_seed(42) | 0.20 | OK |
| 11 | def count_params(module, trainable_only=False): | 0.00 | OK |
| 12 | def compute_metrics(eval_pred): | 0.18 | OK |
| 13 | print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True) | 100.06 | OK |
| 14 | def predict(model, sentence1, sentence2): | 0.12 | OK |
| 15 | from dataclasses import replace | 181.32 | OK |

Сумарний час усіх зафіксованих клітинок: 299.75 с.
